# LEN-Seek quick start

This notebook runs LEN-Seek end to end on CPU in a few minutes:

1. load the released LEN-Seek model (`main_s0`),
2. turn four example binding sites (PDB files in `data/example/queries/`) into model inputs,
3. encode them into per-residue latent point clouds,
4. search a 500-site example database and compare the hits with the paper's ranking,
5. check that a rotated query gives the same encoding,
6. run the same search from the command line.

The example node features (Ankh-base embeddings) are precomputed in `data/example/embeddings/`, so no protein language
model is downloaded here. To embed your own proteins, see the last cell.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("LEN-Seek"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ding-oh/LEN-Seek"], check=True)
    os.chdir("LEN-Seek")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")          # run from the repository root
print("repository root:", os.path.basename(os.getcwd()), "| examples present:", os.path.isdir("data/example"))

In [ ]:
import time
import numpy as np
import torch
import lenseek
from lenseek.assets import fetch
from lenseek.checkpoint import load_model
from lenseek.encode import Encoder
from lenseek.featurize import featurize_site, find_sites
from lenseek.latentdb import LatentDB
from lenseek.search import chamfer_matrix, chamfer_pair, topk

THREADS = int(os.environ.get("LENSEEK_THREADS", "4"))   # CPU threads used below
torch.set_num_threads(THREADS)
print("lenseek", lenseek.__version__, "| torch", torch.__version__)

## 1. Model weights

`fetch` downloads a release asset listed in `lenseek/configs/assets.yaml` into `$LENSEEK_HOME/assets` (default
`~/.cache/lenseek/assets`) and checks its SHA-256. The checkpoint is recognised by its hash, which fixes the model
settings (`lenseek/configs/models.yaml`).

The checkpoint has no public location yet, so `fetch` needs a local copy: place `lenseek_main_s0.pth` in
`$LENSEEK_HOME/assets/`, or set `LENSEEK_ASSET_MIRROR` to a folder that holds it (see `CHECKPOINTS.md`); in Colab,
upload the file first and set `LENSEEK_ASSET_MIRROR` to its folder.

In [ ]:
ckpt = fetch("checkpoint_main_s0")
model, cfg, info = load_model(ckpt)
print(f"{info.name}: epoch {info.epoch}, sha256 {info.sha256[:16]}..., settings from {info.config_source}")
print(info.registry_entry["description"])

## 2. Featurize the example queries

Each site is a pocket file (ATOM records of the pocket residues) next to the protein file. Node features are the
residues' rows of the per-chain Ankh embedding; positions and residue frames come from N, CA and C.

In [ ]:
sites = []
for pocket, protein in find_sites("data/example/queries"):
    s = featurize_site(pocket, protein, "data/example/embeddings")
    sites.append(s)
    print(f"{s.site_id:28s} {len(s):3d} residues  chains {sorted({k[3] for k in s.keys})}  h {s.h.shape}")

## 3. Encode

Every site becomes a cloud of 256-d latent vectors, one per residue (the posterior means). Sites with 15 or fewer
residues use the same kNN graph with self-edges as during training on GPUs.

In [ ]:
t0 = time.time()
encoder = Encoder(model, cfg)
latents = encoder.encode(sites)
print(f"encoded {len(latents)} sites in {time.time() - t0:.1f} s:", [x.shape for x in latents])

## 4. Search the example database

`data/example/db/example_db.npz` holds 500 sites of the released database (114,661 sites): the 50 nearest neighbours
of each example query in the paper's ranking plus random sites. Distances are Chamfer distances between latent
point clouds (smaller = more similar).

In [ ]:
db = LatentDB.load("data/example/db/example_db.npz")
print(db)
t0 = time.time()
dist = chamfer_matrix(latents, db)
print(f"{dist.shape[0]} x {dist.shape[1]} distances in {time.time() - t0:.2f} s\n")
for qi, s in enumerate(sites):
    print(s.site_id)
    for r, j in enumerate(topk(dist[qi], 5), 1):
        print(f"   {r}. {db.site_ids[j]:32s} {dist[qi, j]:8.3f}")

## 5. Compare with the paper's ranking

`data/example/expected_hits.tsv` lists each query's top 10 in the paper's evaluation run (GPU encodings, full
database). The example database contains all of them, so the CPU run here should return the same sites in the same
order, with distances equal up to floating-point noise.

In [ ]:
import csv
expected = {}
for row in csv.DictReader(open("data/example/expected_hits.tsv"), delimiter="\t"):
    expected.setdefault(row["query"], []).append((row["site_id"], float(row["chamfer_distance_paper"])))
for qi, s in enumerate(sites):
    got = [db.site_ids[j] for j in topk(dist[qi], 10)]
    exp_ids = [x[0] for x in expected[s.site_id]]
    diff = max(abs(dist[qi, db.index_of(sid)] - d) for sid, d in expected[s.site_id])
    print(f"{s.site_id:28s} same top-10 order: {got == exp_ids}   max |distance difference| {diff:.2e}")

## 6. Rotation invariance

Edge features use distances, directions expressed in each residue's own frame, and relative orientations, so a rigid
motion of the input must not change the encoding.

In [ ]:
rng = np.random.default_rng(0)
for qi, s in enumerate(sites):
    q, r = np.linalg.qr(rng.normal(size=(3, 3)))
    q = (q * np.sign(np.diag(r))).astype(np.float32)
    if np.linalg.det(q) < 0:
        q[:, 0] *= -1
    shift = rng.normal(size=3).astype(np.float32) * 30
    rotated = (s.h, s.T @ q.T + shift, np.einsum("ij,njk->nik", q, s.R))
    lat_rot = encoder.encode([rotated])[0]
    top_rot = int(np.argmin(chamfer_matrix([lat_rot], db)[0]))
    print(f"{s.site_id:28s} CD(original, rotated) = {chamfer_pair(latents[qi], lat_rot):.1e}   "
          f"top-1 unchanged: {top_rot == int(np.argmin(dist[qi]))}")

## 7. Command line

The same search with the console command. Without `--db`, `lenseek-search` uses the full released database
(fetched once, about 2.8 GB) in its standard edition; `--edition paper --parser legacy` reproduces the paper's rankings exactly.

In [ ]:
out = subprocess.run(["lenseek-search", "data/example/queries", "--embeddings", "data/example/embeddings",
                      "--db", "data/example/db/example_db.npz", "--top-k", "3", "--threads", str(THREADS)],
                     capture_output=True, text=True)
print(out.stdout)
print(out.stderr.strip().splitlines()[-1])

## Your own binding sites

```bash
# protein.pdb + ligand_pocket_1_A.pdb in one directory; --embed computes the Ankh-base embeddings
# (the 2.9 GB weights are downloaded on first use; CC BY-NC-SA 4.0, see the README)
lenseek-search my_protein_dir/ --embed --top-k 100 --out hits.tsv
```

See the README for what a pocket file must contain and `lenseek-search --help` for all options.